In [15]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine

### 1. SQL Server

In [20]:
server = "127.0.0.1,1500"
# database = "DEP2_staging"
database = "DEP2"
username = "sa"
password = "dep2025-G12"
driver = "ODBC Driver 17 for SQL Server"

conn = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)

conn = pyodbc.connect(conn)

### 2. Load FactWifiConnection

In [17]:
wifi_query = """
SELECT f.DateKey, f.TimeKey, f.UserKey, b.SubgroupKey
FROM dbo.FactWifiConnection f
JOIN dbo.BridgeUserSubgroup b
    ON f.UserKey = b.UserKey
"""
wifi_df = pd.read_sql(wifi_query, conn)

C:\Users\Semih\AppData\Local\Temp\ipykernel_4100\2191648252.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  wifi_df = pd.read_sql(wifi_query, conn)


In [18]:
print(wifi_df.head())

    DateKey  TimeKey  UserKey  SubgroupKey
0  20251015   133400       29      5718103
1  20251015   133400       29      5718102
2  20251015   133400       29      5718101
3  20251015   133400       29      5718097
4  20251015   133400       29      5717964


### 3. Aggregate attendance

In [19]:
attendance_df = (
    wifi_df.groupby(['DateKey', 'TimeKey', 'SubgroupKey'])
    .agg(PresentStudents=('UserKey', 'nunique'))
    .reset_index()
)

# attendance_df = attendance_df[attendance_df['SubgroupKey'] == 5816398]
attendance_df

,DateKey,TimeKey,SubgroupKey,PresentStudents
0,20251015,133400,5717941,1
1,20251015,133400,5717964,1
2,20251015,133400,5718097,1
3,20251015,133400,5718101,1
4,20251015,133400,5718102,1
...,...,...,...,...
356537,20251102,115100,5859753,1
356538,20251102,115100,5877942,1
356539,20251102,115100,5877943,1
356540,20251102,115100,5877945,1


In [7]:
# Subgroup linken met SubgroupKey
subgroup_query = """
SELECT SubgroupKey, SubgroupName, SubgroupCode
FROM DEP2.dbo.DimSubgroup
"""
dim_subgroup_df = pd.read_sql(subgroup_query, conn)
dim_subgroup_df

C:\Users\Semih\AppData\Local\Temp\ipykernel_4100\350280976.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_subgroup_df = pd.read_sql(subgroup_query, conn)


,SubgroupKey,SubgroupName,SubgroupCode
0,1,PBA-SO/LOBR/2A,5717692
1,2,PBA-SO/LOBR/2A,5717824
2,3,PBA-SO/LOBR/2A,5717825
3,4,PBA-SO/LOBR/2A,5717826
4,5,PBA-SO/LOBR/2A,5717829
...,...,...,...
6902,6903,GRAD-LABT/DAG/1D,6160650
6903,6904,GRAD-LABT/DAG/1D,6160651
6904,6905,GRAD-LABT/DAG/1D,6160654
6905,6906,GRAD-LABT/DAG/1D,6160659


In [21]:
attendance_df = attendance_df.merge(
    dim_subgroup_df[['SubgroupCode', 'SubgroupName']],
    left_on='SubgroupKey',   
    right_on='SubgroupCode', 
    how='left'
)

attendance_df = attendance_df[['DateKey', 'TimeKey', 'SubgroupKey', 'SubgroupName', 'PresentStudents']]

In [22]:
attendance_df

,DateKey,TimeKey,SubgroupKey,SubgroupName,PresentStudents
0,20251015,133400,5717941,PBA-SO/LOBR/2B,1
1,20251015,133400,5717964,PBA-SO/LOBR/2B,1
2,20251015,133400,5718097,PBA-SO/LOBR/2B,1
3,20251015,133400,5718101,PBA-SO/LOBR/2B,1
4,20251015,133400,5718102,PBA-SO/LOBR/2B,1
...,...,...,...,...,...
356537,20251102,115100,5859753,PBA-AGR-DZ/2E,1
356538,20251102,115100,5877942,PBA-AGR-DZ/2E,1
356539,20251102,115100,5877943,PBA-AGR-DZ/2E,1
356540,20251102,115100,5877945,PBA-AGR-DZ/2E,1
